# IAB Content Classifier — Colab end-to-end

Runs feature extraction and training on a Colab GPU runtime, persisting all outputs to Google Drive so a 12h session timeout doesn't lose work.

**Data layout:** videos are organized **by category folder** on Drive — `iab/<Tier1>/<Tier2>/.../<Leaf>/*.mp4` — where each folder is a taxonomy node name. There is **no hand-written `labels.csv`**: section 3 derives it from the folder tree.

**Before you start:**
1. Set runtime to GPU (Runtime → Change runtime type → T4 / L4 / A100).
2. Have your `iab/` folder tree on Drive at `/content/drive/MyDrive/iab/` (or edit `IAB_DRIVE`).
3. Set `REPO_URL` to your repo (must be public, or the clone will fail).

This notebook does a **test run on the beginning files** (`TEST_LIMIT`) so you can validate the pipeline end-to-end before committing to the full dataset.

## 1. Mount Drive, install deps, clone repo

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Root of the tiered category tree on Drive (contains Attractions/ ... Food & Drink/).
IAB_DRIVE = '/content/drive/MyDrive/Colab Notebooks/iab'
REPO_URL  = 'https://github.com/matthewesp/iab-content-classifier.git'  # <-- set this
REPO_DIR  = '/content/iab-content-classifier'

# Persist HF model weights to Drive so DistilBERT / Whisper / DINO download
# only once across all sessions (~500MB total).
os.environ['HF_HOME'] = f'{IAB_DRIVE}/.hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(f'{IAB_DRIVE}/cache', exist_ok=True)
os.makedirs(f'{IAB_DRIVE}/models', exist_ok=True)

!apt-get -qq install -y ffmpeg
# Note: errors are NOT silenced — if the clone fails (e.g. repo still private),
# you'll see it here instead of a confusing 'no pyproject.toml' later.
![ -d "$REPO_DIR/.git" ] && (cd "$REPO_DIR" && git pull -q) || git clone -q $REPO_URL "$REPO_DIR"
%cd $REPO_DIR
%pip install -q -e .

## 2. Sanity check — device + ASR engine

On Colab GPU you should see `device: cuda` and `amp_enabled: True` (fp16 autocast on backbones). mlx-whisper is Apple-only and is skipped by the env marker in `pyproject.toml`.

In [ ]:
import torch
from video_processor import get_device
print('device:', get_device())
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

# Lightweight backbone init to verify HF model downloads work + autocast is on
from multimodal_backbone import MultimodalBackbone
mb = MultimodalBackbone()
print('amp_enabled:', mb.amp_enabled)

## 3. Generate `labels.csv` from the folder tree (sample across all data)

`scripts/build_labels_from_folders.py` walks `IAB_DRIVE`, maps each video's **deepest containing folder** back to its taxonomy Unique ID (matching the full root→leaf name path), and writes `video_path,leaf_id` rows.

For a test that **spans every category**, use `PER_LEAF` (first N videos per leaf) — this samples the whole taxonomy instead of just the alphabetically-first folders.

- `PER_LEAF = N` → N videos from **each** leaf (representative sample across all data).
- `TEST_LIMIT = N` → hard cap on total rows (applied after per-leaf).
- Set **both to None** for the full dataset.

`--abs` writes absolute Drive paths so `build_features.py` resolves them regardless of CWD.

In [ ]:
# Sample across ALL categories: take the first N videos from every leaf folder.
PER_LEAF   = 2      # videos per leaf — bump up for a bigger test, None to disable
TEST_LIMIT = None   # optional hard cap on total rows; None = no cap

# Build the command in Python (robust to spaces in IAB_DRIVE and optional args)
# instead of relying on shell '$VAR' interpolation in the ! line.
cmd = (
    'python scripts/build_labels_from_folders.py '
    f'--iab "{IAB_DRIVE}" '
    '--taxonomy content_taxonomy_3.1.tsv '
    f'--out "{IAB_DRIVE}/labels.csv" --abs'
)
if PER_LEAF is not None:
    cmd += f' --per-leaf {PER_LEAF}'
if TEST_LIMIT is not None:
    cmd += f' --limit {TEST_LIMIT}'
print(cmd)
!{cmd}

import pandas as pd
_lbl = pd.read_csv(f'{IAB_DRIVE}/labels.csv')
print(f'{len(_lbl)} videos across {_lbl.leaf_id.nunique()} leaves')
_lbl.head(25)

## 4. Feature extraction (the slow stage)

Per-video on T4 is ~10-15s after the first-time HF model download. The cache writes per-video artifacts to Drive, so an interrupted session resumes cheaply — rerun this cell and finished videos hit warm cache.

In [ ]:
!python scripts/build_features.py \
    --labels     "$IAB_DRIVE/labels.csv" \
    --out        "$IAB_DRIVE/features.pt" \
    --taxonomy   content_taxonomy_3.1.tsv \
    --cache-root "$IAB_DRIVE/cache"

## 5. Train classifier

Reads `features.pt`, trains the root router + per-parent routers, writes weights to `$IAB_DRIVE/models/`. Sub-second per router on CUDA.

> On a tiny test slice many parents will have only 1-2 examples — training still runs, but accuracy numbers are meaningless until you scale up the labels. This step is mainly to confirm the train path works.

In [ ]:
!python scripts/train_classifier.py \
    --features "$IAB_DRIVE/features.pt" \
    --out-dir  "$IAB_DRIVE/models" \
    --taxonomy content_taxonomy_3.1.tsv \
    --epochs 50

## 6. (Optional) Smoke classify one video

Loads the trained routers from Drive and runs an end-to-end classification on the first labeled video.

In [ ]:
import json, pandas as pd
from pathlib import Path
from pipeline import VideoClassifier

clf = VideoClassifier(
    router_weights_dir=Path(f'{IAB_DRIVE}/models'),
    cache_root=Path(f'{IAB_DRIVE}/cache'),
)
video = Path(pd.read_csv(f'{IAB_DRIVE}/labels.csv').iloc[0]['video_path'])
print('video:', video.name)
print(json.dumps(clf.classify(video), indent=2))

## Notes

- **Folders are the labels.** Re-run section 3 whenever you add/move videos on Drive; it regenerates `labels.csv` from the current tree.
- **Scale up:** raise/remove `TEST_LIMIT` (or switch to `--per-leaf`) and rerun sections 3→5. Feature cache means already-processed videos are skipped.
- **Disconnects:** rerun cell 1 (re-mount + re-install — fast, wheels cached) then the section you were on. Cache hits skip processed videos.
- **HF downloads:** first feature-extraction run pulls DistilBERT + Whisper-tiny + DINO ViT-S/16 to `$HF_HOME` (~500MB, one-time per Drive).
- **Fast local training:** `features.pt` is small — download from Drive and run `train_classifier.py` on your M1; it's already sub-second.